# Sesión 5 — Gestión de Proyectos y Ruta Crítica

En este notebook aplicaremos **CPM** y **Crashing** al caso del proyecto de apertura del **CEDIS León**.  
La idea es pasar de la tabla de actividades a la red, calcular tiempos y holguras, identificar rutas críticas y evaluar la compresión del proyecto.


## 1. Librerías

Usaremos `pandas` para tablas, `networkx` para la red y `matplotlib` para visualizar los resultados.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product


## 2. Datos del proyecto CEDIS León

Cada actividad tiene una duración normal y una lista de predecesoras.


In [ ]:
actividades = {
    "A": {"descripcion": "Adecuación de infraestructura y obra civil", "duracion": 4, "predecesoras": []},
    "B": {"descripcion": "Montaje de estantería y racks", "duracion": 3, "predecesoras": ["A"]},
    "C": {"descripcion": "Instalación de red eléctrica y servidores", "duracion": 2, "predecesoras": ["A"]},
    "D": {"descripcion": "Recepción e inspección de montacargas", "duracion": 3, "predecesoras": ["B"]},
    "E": {"descripcion": "Configuración y parametrización WMS", "duracion": 4, "predecesoras": ["C"]},
    "F": {"descripcion": "Capacitación de personal operativo", "duracion": 2, "predecesoras": ["B"]},
    "G": {"descripcion": "Pruebas piloto e integración de sistemas", "duracion": 3, "predecesoras": ["D", "E", "F"]},
}

df_actividades = pd.DataFrame([
    {
        "Actividad": a,
        "Descripción": info["descripcion"],
        "Duración": info["duracion"],
        "Predecesoras": ", ".join(info["predecesoras"]) if info["predecesoras"] else "—"
    }
    for a, info in actividades.items()
])

df_actividades


## 3. Construcción de la red AON

La red permite observar las relaciones de precedencia antes de realizar los cálculos del CPM.


In [ ]:
def construir_grafo(datos):
    G = nx.DiGraph()
    for act, info in datos.items():
        G.add_node(act)
        for pred in info["predecesoras"]:
            G.add_edge(pred, act)
    return G

G = construir_grafo(actividades)

pos = {
    "A": (0, 1),
    "B": (1, 1.6),
    "C": (1, 0.4),
    "D": (2, 2.0),
    "F": (2, 1.1),
    "E": (2, 0.2),
    "G": (3, 1.1),
}

plt.figure(figsize=(11, 6))
labels = {a: f"{a}\n{actividades[a]['duracion']} sem" for a in G.nodes}
nx.draw(
    G, pos, labels=labels, with_labels=True,
    node_size=2600, arrowsize=22, font_size=10,
    node_color="lightgray", edge_color="gray"
)
plt.title("Red AON — Proyecto CEDIS León")
plt.axis("off")
plt.show()


## 4. Cálculo del CPM

Primero realizamos el **Forward Pass** para obtener ES y EF. Después hacemos el **Backward Pass** para obtener LS y LF y calcular la holgura.


In [ ]:
def calcular_cpm(datos):
    G = construir_grafo(datos)
    orden = list(nx.topological_sort(G))

    ES, EF = {}, {}

    # Forward Pass
    for act in orden:
        preds = list(G.predecessors(act))
        ES[act] = max((EF[p] for p in preds), default=0)
        EF[act] = ES[act] + datos[act]["duracion"]

    duracion_proyecto = max(EF.values())

    LS, LF = {}, {}

    # Backward Pass
    for act in reversed(orden):
        sucesoras = list(G.successors(act))
        LF[act] = min((LS[s] for s in sucesoras), default=duracion_proyecto)
        LS[act] = LF[act] - datos[act]["duracion"]

    resultados = []
    for act in orden:
        holgura = LS[act] - ES[act]
        resultados.append({
            "Actividad": act,
            "Duración": datos[act]["duracion"],
            "ES": ES[act],
            "EF": EF[act],
            "LS": LS[act],
            "LF": LF[act],
            "Holgura": holgura,
            "Crítica": "Sí" if abs(holgura) < 1e-9 else "No"
        })

    return pd.DataFrame(resultados), duracion_proyecto, G

tabla_cpm, duracion_normal, G = calcular_cpm(actividades)

print(f"Duración total del proyecto: {duracion_normal} semanas")
tabla_cpm


## 5. Ruta crítica

Las actividades con **holgura igual a cero** son críticas. Ahora identificamos las rutas completas que determinan la duración del proyecto.


In [ ]:
def rutas_inicio_fin(G):
    inicios = [n for n in G.nodes if G.in_degree(n) == 0]
    finales = [n for n in G.nodes if G.out_degree(n) == 0]
    rutas = []
    for i in inicios:
        for f in finales:
            rutas.extend(nx.all_simple_paths(G, i, f))
    return rutas

def resumen_rutas(datos):
    G = construir_grafo(datos)
    filas = []
    for ruta in rutas_inicio_fin(G):
        dur = sum(datos[a]["duracion"] for a in ruta)
        filas.append({"Ruta": " → ".join(ruta), "Duración": dur})
    df = pd.DataFrame(filas).sort_values("Duración", ascending=False).reset_index(drop=True)
    max_d = df["Duración"].max()
    df["Crítica"] = np.where(df["Duración"] == max_d, "Sí", "No")
    return df

df_rutas = resumen_rutas(actividades)
df_rutas


In [ ]:
rutas_criticas = df_rutas.loc[df_rutas["Crítica"] == "Sí", "Ruta"].tolist()

print("Rutas críticas:")
for r in rutas_criticas:
    print("•", r)


## 6. Visualización de las rutas críticas

Marcamos las actividades y conexiones críticas para distinguirlas de las ramas con holgura.


In [ ]:
criticas = set(tabla_cpm.loc[tabla_cpm["Crítica"] == "Sí", "Actividad"])

edge_colors = []
edge_widths = []
for u, v in G.edges:
    if u in criticas and v in criticas:
        edge_colors.append("red")
        edge_widths.append(3)
    else:
        edge_colors.append("gray")
        edge_widths.append(1.5)

node_colors = ["mistyrose" if n in criticas else "lightgray" for n in G.nodes]

plt.figure(figsize=(11, 6))
labels = {
    a: f"{a}\n{actividades[a]['duracion']} sem\nHT={int(tabla_cpm.set_index('Actividad').loc[a, 'Holgura'])}"
    for a in G.nodes
}
nx.draw(
    G, pos, labels=labels, with_labels=True,
    node_size=2900, arrowsize=22, font_size=9,
    node_color=node_colors, edge_color=edge_colors, width=edge_widths
)
plt.title(f"CPM — Duración del proyecto: {duracion_normal} semanas")
plt.axis("off")
plt.show()


## 7. Datos de compresión y Cost Slope

Incorporamos el tiempo acelerado y los costos normal y acelerado de cada actividad.


In [ ]:
crashing = pd.DataFrame({
    "Actividad": ["A", "B", "C", "D", "E", "F", "G"],
    "TN": [4, 3, 2, 3, 4, 2, 3],
    "TA": [3, 2, 1, 2, 3, 1, 2],
    "Costo_Normal": [40000, 25000, 15000, 20000, 30000, 10000, 18000],
    "Costo_Acelerado": [46000, 28000, 17000, 24000, 35000, 12000, 25000]
})

crashing["Reducción_Máxima"] = crashing["TN"] - crashing["TA"]
crashing["Cost_Slope"] = (
    (crashing["Costo_Acelerado"] - crashing["Costo_Normal"]) /
    crashing["Reducción_Máxima"]
)

crashing


In [ ]:
costo_normal_total = crashing["Costo_Normal"].sum()
print(f"Costo directo normal del proyecto: ${costo_normal_total:,.0f} USD")


## 8. Primera iteración de Crashing

El proyecto normal tiene dos rutas críticas. Para reducir una semana debemos afectar **ambas rutas** o comprimir una actividad común.


In [ ]:
# Alternativas mostradas en clase para pasar de 13 a 12 semanas
opciones_iteracion_1 = pd.DataFrame({
    "Opción": ["Comprimir A", "Comprimir G", "Comprimir B + C"],
    "Costo adicional": [6000, 7000, 3000 + 2000]
})

opciones_iteracion_1.sort_values("Costo adicional")


In [ ]:
# Aplicamos la opción de menor costo: B y C
actividades_12 = {a: {**info, "predecesoras": list(info["predecesoras"])} for a, info in actividades.items()}
actividades_12["B"]["duracion"] = 2
actividades_12["C"]["duracion"] = 1

tabla_12, duracion_12, G12 = calcular_cpm(actividades_12)
print(f"Nueva duración: {duracion_12} semanas")
resumen_rutas(actividades_12)


## 9. Segunda iteración de Crashing

Con B y C ya comprimidas, comparamos las alternativas disponibles para reducir otra semana.


In [ ]:
opciones_iteracion_2 = pd.DataFrame({
    "Opción": ["Comprimir A", "Comprimir G", "Comprimir D + E"],
    "Costo adicional": [6000, 7000, 4000 + 5000]
})

opciones_iteracion_2.sort_values("Costo adicional")


In [ ]:
# Se comprime A una semana adicional
actividades_11 = {a: {**info, "predecesoras": list(info["predecesoras"])} for a, info in actividades_12.items()}
actividades_11["A"]["duracion"] = 3

tabla_11, duracion_11, G11 = calcular_cpm(actividades_11)

costo_12 = costo_normal_total + 5000
costo_11 = costo_12 + 6000

print(f"Duración final: {duracion_11} semanas")
print(f"Costo directo final: ${costo_11:,.0f} USD")
print(f"Costo adicional: ${costo_11 - costo_normal_total:,.0f} USD")

tabla_11


## 10. Comparación del proyecto normal y acelerado

Comparamos duración, costo y rutas críticas antes y después de la compresión.


In [ ]:
comparacion = pd.DataFrame({
    "Escenario": ["Normal", "Después de Iteración 1", "Después de Iteración 2"],
    "Duración (semanas)": [duracion_normal, duracion_12, duracion_11],
    "Costo directo (USD)": [costo_normal_total, costo_12, costo_11],
    "Costo adicional (USD)": [0, 5000, 11000]
})

comparacion


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(comparacion["Duración (semanas)"], comparacion["Costo directo (USD)"], marker="o")
plt.xlabel("Duración del proyecto (semanas)")
plt.ylabel("Costo directo (USD)")
plt.title("Relación tiempo–costo del proyecto")
plt.grid(alpha=0.25)
plt.gca().invert_xaxis()
plt.show()


## 11. Red del proyecto acelerado

La red final permite comprobar que las dos rutas críticas se mantienen después de las compresiones realizadas.


In [ ]:
criticas_11 = set(tabla_11.loc[tabla_11["Crítica"] == "Sí", "Actividad"])

edge_colors = []
edge_widths = []
for u, v in G11.edges:
    if u in criticas_11 and v in criticas_11:
        edge_colors.append("red")
        edge_widths.append(3)
    else:
        edge_colors.append("gray")
        edge_widths.append(1.5)

node_colors = ["mistyrose" if n in criticas_11 else "lightgray" for n in G11.nodes]

plt.figure(figsize=(11, 6))
labels = {
    a: f"{a}\n{actividades_11[a]['duracion']} sem\nHT={int(tabla_11.set_index('Actividad').loc[a, 'Holgura'])}"
    for a in G11.nodes
}
nx.draw(
    G11, pos, labels=labels, with_labels=True,
    node_size=2900, arrowsize=22, font_size=9,
    node_color=node_colors, edge_color=edge_colors, width=edge_widths
)
plt.title(f"Proyecto acelerado — Duración: {duracion_11} semanas")
plt.axis("off")

## 12. Función reutilizable

Esta función permite modificar duraciones y volver a calcular automáticamente el CPM.


In [ ]:
def evaluar_proyecto(duraciones):
    datos = {
        a: {
            **info,
            "duracion": duraciones.get(a, info["duracion"]),
            "predecesoras": list(info["predecesoras"])
        }
        for a, info in actividades.items()
    }
    tabla, duracion, _ = calcular_cpm(datos)
    rutas = resumen_rutas(datos)
    return tabla, duracion, rutas

# Ejemplo
tabla_ejemplo, duracion_ejemplo, rutas_ejemplo = evaluar_proyecto({"D": 2, "E": 3})
print("Duración:", duracion_ejemplo)
rutas_ejemplo


# Actividad para el alumno

Una empresa prepara la **implementación de un nuevo centro de distribución regional**. A partir de los datos siguientes:

| Act. | Actividad | Duración | Predecesoras |
|---|---|---:|---|
| A | Diseño del layout | 3 | — |
| B | Preparación del almacén | 5 | A |
| C | Configuración del sistema | 4 | A |
| D | Instalación de equipos | 3 | B |
| E | Integración de información | 2 | C |
| F | Pruebas operativas | 4 | D, E |

Realiza en Python:

1. Construye la red AON.
2. Calcula ES, EF, LS y LF.
3. Calcula la holgura de cada actividad.
4. Identifica la ruta crítica y la duración total.
5. Supón que **B puede reducirse de 5 a 4 semanas por $4,000 USD** y **C de 4 a 3 semanas por $2,500 USD**. Determina si comprimir alguna de estas actividades reduce la duración total.
6. Explica brevemente cuál decisión recomendarías.


In [ ]:
# Datos iniciales para resolver la actividad
ejercicio = {
    "A": {"descripcion": "Diseño del layout", "duracion": 3, "predecesoras": []},
    "B": {"descripcion": "Preparación del almacén", "duracion": 5, "predecesoras": ["A"]},
    "C": {"descripcion": "Configuración del sistema", "duracion": 4, "predecesoras": ["A"]},
    "D": {"descripcion": "Instalación de equipos", "duracion": 3, "predecesoras": ["B"]},
    "E": {"descripcion": "Integración de información", "duracion": 2, "predecesoras": ["C"]},
    "F": {"descripcion": "Pruebas operativas", "duracion": 4, "predecesoras": ["D", "E"]},
}

# Continúa aquí...


## Cierre

El CPM permite identificar qué actividades controlan la duración del proyecto.  
El **Crashing** agrega la dimensión económica: no basta con saber qué actividad puede acelerarse, también debemos evaluar **cuánto cuesta reducir el tiempo total**.
